# Notebook 03: Cleaning and Splits

**Project:** Credit Decision Assistant

## What this notebook is for

Notebook 02 decided *what* the model predicts. This notebook decides *what the model is allowed to look at*, and *which loans it learns from versus which loans it is tested on*.

It answers three questions:

1. **Which of the 151 columns would a lender actually have when someone applies for a loan?** Only those can be used.
2. **Are those columns usable?** Some only exist in certain years, some are mostly empty for good reasons, and some contain text or impossible values that need tidying.
3. **How should the loans be split?** The model must be judged on loans issued *after* the ones it learned from, just like in real life.

## A few terms used in this notebook

| Term | What it means here |
|---|---|
| **Feature** | A piece of information the model uses to make a prediction, such as income or credit score. |
| **Data leakage** | Letting the model see information it wouldn't have at the moment of the real decision. It makes a model look brilliant in testing and useless in practice. |
| **Train / validation / test set** | The loans the model learns from, the loans used to tune it, and the loans kept aside to give an honest final score. |
| **Imputation** | Filling in missing values with an estimate. Not done here; it's left to later notebooks so it can be fitted on the training set only. |
| **Structural missingness** | A value that is missing because of what's true, not because data was lost. For example "months since last bankruptcy" is empty for someone who has never been bankrupt. |
| **Selection bias** | When the data only contains a filtered group (here, approved borrowers), so patterns in it can differ from patterns in the full population. |
| **Redlining** | Refusing or overcharging people based on where they live. It's illegal in US lending because location can stand in for race or ethnicity. |

## What this notebook produces

- `data/processed/train.parquet`, `valid.parquet`, `test.parquet`: the three time-based splits, with 62 features and the target.
- An updated `audit_config.json` with the split dates, the feature lists and the reasons columns were excluded.

## Setup

Paths, the settings from Notebooks 01 and 02, and the target table (1,709,332 eligible loans with their 12-month default flag).

In [1]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT  = Path.cwd().parent           # notebooks/ is one level below the project folder
INTERIM_DIR   = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR       = PROJECT_ROOT / "outputs" / "figures"

# Settings from Notebooks 01 and 02 (snapshot, scope, target definition)
with open(INTERIM_DIR / "audit_config.json") as f:
    config = json.load(f)

RAW_PARQUET = INTERIM_DIR / "loans_raw.parquet"                  # all 151 columns, built in Notebook 01
target = pd.read_parquet(PROCESSED_DIR / "target_12m.parquet")   # eligible loans + 12-month default flag
print(f"Target table: {len(target):,} loans | default rate {target['default_12m'].mean():.2%}")

Target table: 1,709,332 loans | default rate 3.52%


## Step 1: Sort every column by when its information becomes known

The rule for this whole project: **the model may only use information a lender has at the moment someone applies.**

The clearest example of breaking that rule is `total_rec_prncp`, the amount of the loan repaid so far. It predicts default extremely well, because people who defaulted repaid less. But it's only known *after* the loan has been running. A model trained with it would score brilliantly in testing and then be useless on a real application, where that number doesn't exist yet. That's **data leakage**.

So every one of the 151 columns goes into one of ten groups:

| Group | Columns | Examples | Used? | Why |
|---|---|---|---|---|
| Key and timing | 2 | `id`, `issue_d` | As identifiers only | Needed to join tables and split by time |
| After issue | 39 | payments received, recoveries, latest credit score, hardship plans, settlements | No | Only known after the loan starts: leakage |
| Lender decision | 8 | grade, interest rate, monthly payment, amount funded | No | Lending Club's own verdict and pricing (Notebook 01, Step 9) |
| Admin | 3 | `member_id`, `url`, `policy_code` | No | Empty or constant |
| Free text | 3 | `desc`, `title`, `emp_title` | No (for now) | Mostly missing, inconsistent, or thousands of unique job titles |
| Location | 2 | `zip_code`, `addr_state` | No | See below |
| Joint application | 17 | co-applicant's income, credit score | No | Only individual applications are in scope |
| Application form | 8 | loan amount, term, income, debt-to-income, purpose, home ownership, employment length, income verified | Yes | What the applicant tells the lender |
| Credit bureau (core) | 55 | credit score, card usage, number of accounts, past late payments, recent credit checks | Yes | What the credit report says at application |
| Credit bureau (extended) | 14 | `open_acc_6m`, `all_util`, `inq_fi` | To be checked | Probably only exist from 2016 (tested in Step 3) |

**Why location is left out.** Postcode and state can act as a hidden stand-in for race or ethnicity, because many neighbourhoods are strongly segregated. Lending decisions based on where people live (redlining) are illegal in US lending. Lending Club's data has no direct information on race, gender or age, and leaving location out stops the model from rebuilding that information indirectly. It costs a little accuracy and makes the model fairer and easier to defend.

**Two borderline calls:**
- `verification_status` (whether Lending Club checked the stated income) is part of Lending Club's process, but a bank would know it when deciding too, so it stays.
- `funded_amnt` is how much Lending Club actually lent, which is Lending Club's decision. The *requested* amount, `loan_amnt`, is what the applicant asks for, so that's the one used.

The core bureau group isn't typed out by hand. It's defined as "every column not listed anywhere else", and two checks confirm that no column is in two groups and none was forgotten or misspelled. That way a leakage column can't slip into the features by accident.

**Result:** all **151 columns** accounted for, leaving **77 candidate features** (8 application, 55 core bureau, 14 extended bureau).

In [2]:
# Every column in the raw data, sorted by when its information becomes known
COLUMN_GROUPS = {
    "key_and_timing": ["id", "issue_d"],
    "after_issue": [
        "loan_status", "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv",
        "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries", "collection_recovery_fee",
        "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d",
        "last_fico_range_high", "last_fico_range_low", "pymnt_plan",
        "hardship_flag", "hardship_type", "hardship_reason", "hardship_status", "deferral_term",
        "hardship_amount", "hardship_start_date", "hardship_end_date", "payment_plan_start_date",
        "hardship_length", "hardship_dpd", "hardship_loan_status",
        "orig_projected_additional_accrued_interest", "hardship_payoff_balance_amount",
        "hardship_last_payment_amount", "debt_settlement_flag", "debt_settlement_flag_date",
        "settlement_status", "settlement_date", "settlement_amount", "settlement_percentage", "settlement_term",
    ],
    "lender_decision": ["grade", "sub_grade", "int_rate", "installment", "funded_amnt", "funded_amnt_inv",
                        "initial_list_status", "disbursement_method"],
    "admin": ["member_id", "url", "policy_code"],
    "free_text": ["desc", "title", "emp_title"],
    "location": ["zip_code", "addr_state"],
    "joint_application": [
        "application_type", "annual_inc_joint", "dti_joint", "verification_status_joint", "revol_bal_joint",
        "sec_app_fico_range_low", "sec_app_fico_range_high", "sec_app_earliest_cr_line",
        "sec_app_inq_last_6mths", "sec_app_mort_acc", "sec_app_open_acc", "sec_app_revol_util",
        "sec_app_open_act_il", "sec_app_num_rev_accts", "sec_app_chargeoff_within_12_mths",
        "sec_app_collections_12_mths_ex_med", "sec_app_mths_since_last_major_derog",
    ],
    "application": ["loan_amnt", "term", "emp_length", "home_ownership", "annual_inc",
                    "verification_status", "purpose", "dti"],
    "bureau_extended": ["open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m", "mths_since_rcnt_il",
                        "total_bal_il", "il_util", "open_rv_12m", "open_rv_24m", "max_bal_bc", "all_util",
                        "inq_fi", "total_cu_tl", "inq_last_12m"],
}

# Every other column is a core credit bureau field; work it out rather than typing all 55
with open(INTERIM_DIR / "loans_raw_schema.json") as f:
    schema = json.load(f)
all_cols = schema["numeric_cols"] + schema["text_cols"]
listed = [c for cols in COLUMN_GROUPS.values() for c in cols]
COLUMN_GROUPS["bureau_core"] = [c for c in all_cols if c not in listed]

# Checks: no column in two groups, no made-up names, nothing left out
listed = [c for cols in COLUMN_GROUPS.values() for c in cols]
assert len(listed) == len(set(listed)), "A column appears in more than one group"
assert set(listed) == set(all_cols), "Group lists don't match the columns in the file"

summary = pd.DataFrame({"n_columns": {g: len(c) for g, c in COLUMN_GROUPS.items()}})
summary["used_as_features"] = summary.index.isin(["application", "bureau_core", "bureau_extended"])
display(summary)
print(f"Total: {summary['n_columns'].sum()} columns")
print("\nCore bureau columns:", COLUMN_GROUPS["bureau_core"])

,n_columns,used_as_features
key_and_timing,2,False
after_issue,39,False
lender_decision,8,False
admin,3,False
free_text,3,False
location,2,False
joint_application,17,False
application,8,True
bureau_extended,14,True
bureau_core,55,True


Total: 151 columns

Core bureau columns: ['delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_l

## Step 2: Load the eligible loans with their candidate features

The raw Parquet file is read **one block at a time**. From each block only the eligible loans and the 77 candidate columns are kept, and numbers are stored as `float32` instead of `float64`. That halves the memory while still keeping about 7 significant digits, which is plenty for values like incomes or credit limits. Then the target from Notebook 02 is attached to each loan by its ID.

Two safety checks:
- `validate="one_to_one"` makes pandas stop with an error if any loan ID appears twice, so the join can't quietly duplicate loans.
- Every eligible loan must have found its features. `loan_amnt` is never missing in the raw data, so a missing value there would mean a failed match.

**Result:** **1,709,332 loans x 84 columns** (77 candidate features plus IDs, dates and the three versions of the target), using only **0.70 GB**. Every eligible loan was matched. It took about 55 seconds, mostly spent checking each row's ID against the list of eligible loans.

In [3]:
FEATURE_CANDIDATES = (COLUMN_GROUPS["application"]
                      + COLUMN_GROUPS["bureau_core"]
                      + COLUMN_GROUPS["bureau_extended"])
print(f"Candidate feature columns: {len(FEATURE_CANDIDATES)}")

eligible_ids = target["id"].values          # the 1,709,332 loans from Notebook 02

# Read the raw file block by block so memory never holds more than one block of raw data
pf = pq.ParquetFile(RAW_PARQUET)
parts = []
t0 = time.time()
for i in range(pf.num_row_groups):
    # Read one block of rows, only the columns we need
    chunk = pf.read_row_group(i, columns=["id"] + FEATURE_CANDIDATES).to_pandas()
    chunk = chunk[chunk["id"].isin(eligible_ids)]            # keep eligible loans only
    num_cols = chunk.select_dtypes("float64").columns
    chunk[num_cols] = chunk[num_cols].astype("float32")      # halve the memory of numeric columns
    parts.append(chunk)
features = pd.concat(parts, ignore_index=True)
del parts                                      # free the list of blocks once combined
print(f"Loaded {len(features):,} loans x {features.shape[1]} columns in {time.time() - t0:.1f}s "
      f"| memory {features.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Attach the target and timing columns. loan_amnt already comes from the raw file, so take
# only what the features table doesn't have, to avoid duplicate columns
TARGET_KEEP = ["id", "issue_d", "issue_year", "term_m",
               "default_12m", "default_12m_lenient", "default_12m_strict"]
df = target[TARGET_KEEP].merge(features, on="id", how="left", validate="one_to_one")
del features                                   # free memory: everything is now in df

# Every eligible loan must have found its features
assert len(df) == len(target), "Row count changed during the join"
assert df["loan_amnt"].notna().all(), "Some eligible loans have no feature data"
print(f"Model table: {df.shape[0]:,} rows x {df.shape[1]} columns")

Candidate feature columns: 77
Loaded 1,709,332 loans x 78 columns in 55.1s | memory 0.70 GB
Model table: 1,709,332 rows x 84 columns


## Step 3: Split by time, and check which columns exist in each period

**The split.** The model has to predict loans issued *after* the ones it learned from, so the split follows the calendar, never a random shuffle. A random split would let the model learn from 2017 loans and then be tested on 2015 loans, which is a kind of looking into the future that inflates results.

| Set | Issue dates | Loans | 12-month default rate | Purpose |
|---|---|---|---|---|
| Train | Jan 2013 to Dec 2015 | 791,027 | 3.11% | The model learns from these |
| Validation | Jan 2016 to Dec 2016 | 425,618 | 4.01% | Tuning settings and adjusting the probability level |
| Test | Jan 2017 to Mar 2018 | 492,687 | 3.74% | Used once, at the very end, for an honest final score |

The test set stays untouched until the end. Every time results on it are checked and something gets adjusted because of them, it becomes a little less of a fair test.

Notice that the default rate is higher in validation and test than in train, as Notebook 02 showed. A model trained on 2013 to 2015 will tend to underestimate the overall level of risk later on, which is exactly why the validation year is used to recalibrate it.

**The completeness check.** For each candidate column, the share of missing values in each period, with two rules:
- **More than 50% missing in train:** candidate for dropping, since the model can't learn much from a mostly empty column.
- **Missing share changes by more than 10 percentage points between train and test:** flagged, because that's the "the column tells you the year" problem from Notebook 01.

**Result:** 19 columns hit the first rule and 14 hit the second. They fall into two very different groups:

1. **All 14 extended bureau columns** are about 97% missing in train and almost complete in validation and test. They're missing **because of time**: Lending Club only started collecting them at the very end of 2015. Keeping them would let the model learn "missing means an older loan". They're dropped.
2. **Five "months since..." columns** are 50% to 83% missing, but at the **same rate in every period**. They're missing for a different reason, covered next.

The remaining gaps are small (1% to 12%) and stable over time: no recent credit check (`mths_since_recent_inq`), never had an instalment loan (`mo_sin_old_il_acct`), no credit card (the `bc_*` columns), and employment length left blank. They're kept and filled in later, using the training set only.

In [4]:
# Split by issue date: learn from the past, test on the future
TRAIN_END = pd.Timestamp("2015-12-01")         # train: Jan 2013 to Dec 2015
VALID_END = pd.Timestamp("2016-12-01")         # validation: 2016, test: Jan 2017 to Mar 2018

df["split"] = np.select(
    [df["issue_d"] <= TRAIN_END, df["issue_d"] <= VALID_END],
    ["train", "valid"],
    default="test",
)
split_summary = df.groupby("split").agg(
    n_loans      = ("id", "size"),
    issued_from  = ("issue_d", "min"),
    issued_to    = ("issue_d", "max"),
    default_rate = ("default_12m", "mean"),
).loc[["train", "valid", "test"]]
display(split_summary.style.format({"n_loans": "{:,}", "default_rate": "{:.2%}",
                                    "issued_from": "{:%b-%Y}", "issued_to": "{:%b-%Y}"}))

# Share of missing values per column, in each period
missing = (df[FEATURE_CANDIDATES].isna()
             .groupby(df["split"]).mean().T[["train", "valid", "test"]])
missing["group"] = ["application" if c in COLUMN_GROUPS["application"]
                    else "bureau_extended" if c in COLUMN_GROUPS["bureau_extended"]
                    else "bureau_core" for c in missing.index]
# Rule 1: mostly empty when the model learns. Rule 2: completeness changes over time
missing["drop_mostly_empty"]  = missing["train"] > 0.50
missing["flag_shift"]         = (missing["test"] - missing["train"]).abs() > 0.10

print("Columns with any missing values, most missing in train first:")
show = missing[(missing[["train", "valid", "test"]] > 0).any(axis=1)].sort_values("train", ascending=False)
display(show.style.format({"train": "{:.1%}", "valid": "{:.1%}", "test": "{:.1%}"}))

print(f"\nDrop (over 50% missing in train): {missing['drop_mostly_empty'].sum()}")
print(f"Flag (missing share shifts over 10 points): {missing['flag_shift'].sum()}")

,n_loans,issued_from,issued_to,default_rate
split,,,,
train,"791,027",Jan-2013,Dec-2015,3.11%
valid,"425,618",Jan-2016,Dec-2016,4.01%
test,"492,687",Jan-2017,Mar-2018,3.74%


Columns with any missing values, most missing in train first:


split,train,valid,test,group,drop_mostly_empty,flag_shift
il_util,97.7%,13.2%,14.2%,bureau_extended,True,True
mths_since_rcnt_il,97.4%,2.7%,3.1%,bureau_extended,True,True
inq_last_12m,97.3%,0.0%,0.0%,bureau_extended,True,True
open_rv_24m,97.3%,0.0%,0.0%,bureau_extended,True,True
open_acc_6m,97.3%,0.0%,0.0%,bureau_extended,True,True
open_il_24m,97.3%,0.0%,0.0%,bureau_extended,True,True
total_bal_il,97.3%,0.0%,0.0%,bureau_extended,True,True
open_il_12m,97.3%,0.0%,0.0%,bureau_extended,True,True
open_rv_12m,97.3%,0.0%,0.0%,bureau_extended,True,True
max_bal_bc,97.3%,0.0%,0.0%,bureau_extended,True,True



Drop (over 50% missing in train): 19
Flag (missing share shifts over 10 points): 14


## Step 4: Two kinds of missing data

The extended bureau block is dropped here, for the reason above.

The five "months since..." columns need a closer look:

| Column | Measures months since... | Missing in train / valid / test |
|---|---|---|
| `mths_since_last_record` | a public record, such as a bankruptcy or court judgement | 83% / 81% / 83% |
| `mths_since_recent_bc_dlq` | the last late payment on a credit card | 75% / 74% / 77% |
| `mths_since_last_major_derog` | the last serious negative event (90+ days late) | 73% / 71% / 73% |
| `mths_since_recent_revol_delinq` | the last late payment on any revolving account | 65% / 64% / 67% |
| `mths_since_last_delinq` | the last late payment of any kind | 50% / 47% / 51% |

The likely reason they're missing: **the event never happened.** You can't have "months since your last bankruptcy" if you've never been bankrupt. If that's right, missing values carry real information and shouldn't be thrown away just because there are a lot of them.

**Testing that reading against the data.** Other columns *count* the same events. If `delinq_2yrs` (late payments in the last 2 years) is above zero, the person has had a late payment, so `mths_since_last_delinq` should be present and 24 or less. The same logic applies to `pub_rec` (number of public records) and `mths_since_last_record`.

**Result of the check:**
- Public record but `mths_since_last_record` missing: **0.00%**.
- Late payment in the last 2 years but `mths_since_last_delinq` missing: **1.29%**, and **97.8%** of those present are 24 months or less. The small leftovers are the normal untidiness of credit report data.

So missing does mean "never happened" for these columns.

**Does it matter for default? Surprisingly little.** On the training set:

| Column | Default rate: never happened | Default rate: happened |
|---|---|---|
| `mths_since_last_record` | 3.08% | 3.29% |
| `mths_since_last_major_derog` | 3.07% | 3.23% |
| `mths_since_recent_bc_dlq` | 3.12% | 3.10% |
| `mths_since_recent_revol_delinq` | 3.14% | 3.05% |
| `mths_since_last_delinq` | 3.16% | 3.06% |

I expected people with a past late payment or bankruptcy to default noticeably more often. They don't, and for late payments it's even slightly less.

The most likely explanation is **selection bias**. These are only the people Lending Club *approved*. Someone with a past bankruptcy only got through if the rest of their profile was strong enough to make up for it: high income, low debt, a good score. So among approved borrowers, the people with a black mark aren't much riskier overall, because the screening already used that information. It's a real limitation of building a lending model only from approved loans: the model never sees the people who were turned away. Lending Club also published its rejected applications, and using them to correct for this (called *reject inference*) is a possible extension.

**Decision:** keep these five columns for now. The plain yes/no split carries little signal, but *how recently* the event happened, or its combination with other features, might still matter. Notebook 05 measures each feature's predictive value properly on the training set and decides with evidence. Only the training set is used for this kind of judgement, so validation and test stay unseen.

In [5]:
# 1. Drop the extended bureau block: missing because of time, not because of the borrower
df = df.drop(columns=COLUMN_GROUPS["bureau_extended"])
print(f"Dropped {len(COLUMN_GROUPS['bureau_extended'])} extended bureau columns -> {df.shape[1]} columns left")

# 2. Columns where missing most likely means "this never happened"
STRUCTURAL_MISSING = [
    "mths_since_last_record", "mths_since_recent_bc_dlq", "mths_since_last_major_derog",
    "mths_since_recent_revol_delinq", "mths_since_last_delinq",
]

# Test the "never happened" reading against columns that count the same events
has_recent_delinq = df["delinq_2yrs"] > 0          # at least one late payment in the last 2 years
has_public_record = df["pub_rec"] > 0              # at least one public record (e.g. bankruptcy)
print("\nIf someone HAS the event, the 'months since' value should be present:")
print(f"  late payment in last 2 years, but mths_since_last_delinq missing: "
      f"{df.loc[has_recent_delinq, 'mths_since_last_delinq'].isna().mean():.2%}")
print(f"  ...and when present, it is 24 months or less:                     "
      f"{(df.loc[has_recent_delinq, 'mths_since_last_delinq'] <= 24).mean():.2%}")
print(f"  public record, but mths_since_last_record missing:                "
      f"{df.loc[has_public_record, 'mths_since_last_record'].isna().mean():.2%}")

# Is "missing" informative? Compare default rates when the value is missing vs present (train only)
train = df["split"].eq("train")
info = pd.DataFrame({
    "default_rate_if_missing": [df.loc[train & df[c].isna(),  "default_12m"].mean() for c in STRUCTURAL_MISSING],
    "default_rate_if_present": [df.loc[train & df[c].notna(), "default_12m"].mean() for c in STRUCTURAL_MISSING],
}, index=STRUCTURAL_MISSING)
info["ratio_present_vs_missing"] = info["default_rate_if_present"] / info["default_rate_if_missing"]
print("\nDefault rate (train) when the event never happened vs when it did:")
display(info.style.format({"default_rate_if_missing": "{:.2%}", "default_rate_if_present": "{:.2%}",
                           "ratio_present_vs_missing": "{:.2f}x"}))

Dropped 14 extended bureau columns -> 71 columns left

If someone HAS the event, the 'months since' value should be present:
  late payment in last 2 years, but mths_since_last_delinq missing: 1.29%
  ...and when present, it is 24 months or less:                     97.79%
  public record, but mths_since_last_record missing:                0.00%

Default rate (train) when the event never happened vs when it did:


,default_rate_if_missing,default_rate_if_present,ratio_present_vs_missing
mths_since_last_record,3.08%,3.29%,1.07x
mths_since_recent_bc_dlq,3.12%,3.10%,0.99x
mths_since_last_major_derog,3.07%,3.23%,1.05x
mths_since_recent_revol_delinq,3.14%,3.05%,0.97x
mths_since_last_delinq,3.16%,3.06%,0.97x


## Step 5: Turn text into numbers and fix impossible values

Each change here is a repair or a reshaping. Nothing is learned from the data, so it's safe to apply to all three splits at once.

1. **Employment length** is stored as text ("< 1 year", "3 years", "10+ years") and becomes a number of years. "< 1 year" becomes 0 and "10+ years" becomes 10. The 6% left blank stay missing.
2. **Credit history length.** `earliest_cr_line` is the month the borrower opened their first credit account. A date isn't something the model can use directly, so it becomes the number of months between that date and the loan's issue date. A longer history usually means a more established borrower.
3. **Home ownership** has two tiny categories, "ANY" (511 loans) and "NONE" (1 loan), merged into "OTHER".
4. **FICO score** comes as a low and a high end of a band. If the high end is always the low end plus 4, it adds nothing.
5. **Term** exists twice, as text (" 36 months") and as a number from Notebook 02. The two are checked to agree, then the text version is dropped.
6. **Impossible values** are counted and set to missing: zero or negative income, a negative debt-to-income ratio, and a debt-to-income ratio over 100% (debt payments larger than income), which almost certainly means a data error.

Values that are **unusual but possible**, such as very high incomes, aren't touched here. Deciding where to cap them means looking at the distribution, and that has to be done on the training set only.

Finally, a check for categories that appear in validation or test but never in train. The model couldn't learn anything about those.

**Result:**
- Employment length converted; "10+ years" is the largest group at 34%.
- **Credit history length:** no parsing failures and no negative values. The median borrower has 180 months (15 years) of history. The minimum is exactly **36 months**, which matches Lending Club's rule that applicants needed at least 3 years of credit history: the data reflects the platform's own policy. The maximum, 999 months (83 years), is extreme but possible.
- **Impossible values:** only 1 loan had a negative debt-to-income ratio, now missing. No zero incomes and no ratios over 100% in this population. The extreme values seen in the full file belonged to joint applications, which are out of scope.
- **No unseen categories** in validation or test.
- **FICO:** the gap isn't always 4. 232 loans have a gap of 5, so `fico_range_high` wasn't dropped automatically. The next step looks at why.

In [6]:
# 1. Employment length: text -> years ("< 1 year" -> 0, "10+ years" -> 10, missing stays missing)
print("emp_length values:")
print(df["emp_length"].value_counts(dropna=False).to_string())
emp_years = df["emp_length"].str.extract(r"(\d+)")[0].astype("float32")
emp_years[df["emp_length"].eq("< 1 year")] = 0          # "< 1 year" contains a 1, so fix it explicitly
df["emp_length_yrs"] = emp_years
df = df.drop(columns=["emp_length"])

# 2. Credit history length: months from the first credit account to the loan's issue date
first_credit = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")
df["credit_history_m"] = ((df["issue_d"].dt.year - first_credit.dt.year) * 12
                          + (df["issue_d"].dt.month - first_credit.dt.month)).astype("float32")
print(f"\nCredit history length (months): failed to parse {first_credit.isna().sum() - df['earliest_cr_line'].isna().sum()}, "
      f"negative values {(df['credit_history_m'] < 0).sum()}")
print(df["credit_history_m"].describe().round(1).to_string())
df = df.drop(columns=["earliest_cr_line"])

# 3. Home ownership: merge the tiny categories
print("\nhome_ownership before:")
print(df["home_ownership"].value_counts().to_string())
df["home_ownership"] = df["home_ownership"].replace({"ANY": "OTHER", "NONE": "OTHER"})

# 4. FICO: is the high end always the low end + 4?
fico_gap = (df["fico_range_high"] - df["fico_range_low"]).value_counts()
print("\nfico_range_high - fico_range_low:")
print(fico_gap.to_string())
if len(fico_gap) == 1:
    df = df.drop(columns=["fico_range_high"])
    print("-> constant gap, dropped fico_range_high")

# 5. Term: the text version must agree with the number from Notebook 02, then drop it
term_from_text = df["term"].str.extract(r"(\d+)")[0].astype("int8")
assert (term_from_text == df["term_m"]).all(), "term text and term_m disagree"
df = df.drop(columns=["term"])

# 6. Impossible values: count them, then set them to missing
impossible = {
    "annual_inc <= 0": df["annual_inc"] <= 0,
    "dti < 0":         df["dti"] < 0,
    "dti > 100":       df["dti"] > 100,
}
for name, mask in impossible.items():
    print(f"{name:<16} {mask.sum():>7,} loans")
df.loc[df["annual_inc"] <= 0, "annual_inc"] = np.nan
df.loc[(df["dti"] < 0) | (df["dti"] > 100), "dti"] = np.nan

# Categories that appear in validation/test but never in train: the model couldn't have learned them
for col in ["home_ownership", "verification_status", "purpose"]:
    unseen = set(df.loc[~train, col].dropna()) - set(df.loc[train, col].dropna())
    print(f"{col}: categories not seen in train -> {sorted(unseen) if unseen else 'none'}")

print(f"\nColumns now: {df.shape[1]}")

emp_length values:
emp_length
10+ years    581795
2 years      154898
3 years      137022
< 1 year     127066
1 year       112154
5 years      104559
NaN          102107
4 years      101667
6 years       77231
8 years       73649
7 years       72116
9 years       65068

Credit history length (months): failed to parse 0, negative values 0
count   1,709,332.0000
mean          198.9000
std            91.7000
min            36.0000
25%           137.0000
50%           180.0000
75%           246.0000
max           999.0000

home_ownership before:
home_ownership
MORTGAGE    835151
RENT        680686
OWN         192983
ANY            511
NONE             1

fico_range_high - fico_range_low:
4.0000    1709100
5.0000        232
annual_inc <= 0        0 loans
dti < 0                1 loans
dti > 100              0 loans
home_ownership: categories not seen in train -> none
verification_status: categories not seen in train -> none
purpose: categories not seen in train -> none

Columns now: 70


## Step 6: Final check on FICO, then save

**FICO.** Credit scores top out at 850, so the highest band is 845 to 850, which is 5 points wide instead of 4. If all 232 gap-of-5 loans sit in that top band, `fico_range_high` still adds nothing.

**Result:** all 232 have a low end of 845, so `fico_range_high` is dropped.

**Saving.** Each split goes into its own file. Keeping the test set separate makes it much harder to load it by accident, so it stays untouched until the end. The feature lists, which features are categories, the five "never happened" columns and the split dates all go into the config file, so later notebooks read them instead of retyping them.

**Result:**
- **62 features:** 59 numeric and 3 categorical (`home_ownership`, `verification_status`, `purpose`).
- Train: 791,027 loans (3.11% default rate, 52 MB)
- Validation: 425,618 loans (4.01%, 29 MB)
- Test: 492,687 loans (3.74%, 34 MB)

In [7]:
# 1. FICO: are the gap-of-5 loans all in the top band (845-850)?
gap5 = (df["fico_range_high"] - df["fico_range_low"]) == 5
print(f"Loans with a gap of 5: {gap5.sum()} | their fico_range_low values: {sorted(df.loc[gap5, 'fico_range_low'].unique())}")
if (df.loc[gap5, "fico_range_low"] == 845).all():
    df = df.drop(columns=["fico_range_high"])
    print("-> all in the 845-850 top band: fico_range_high adds nothing, dropped")

# 2. Final feature list: everything except identifiers, timing, split label and targets
NON_FEATURES = ["id", "issue_d", "issue_year", "split",
                "default_12m", "default_12m_lenient", "default_12m_strict"]
FEATURES     = [c for c in df.columns if c not in NON_FEATURES]
# Text columns can show up as "object" or as pandas 3's "str" type, so check for both
CATEGORICAL  = [c for c in FEATURES if df[c].dtype == "object" or str(df[c].dtype) in ("str", "string")]
NUMERIC      = [c for c in FEATURES if c not in CATEGORICAL]
print(f"\nFeatures: {len(FEATURES)}  ({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)")
print("Categorical:", CATEGORICAL)

# 3. Save each split to its own file (the split label is dropped: the file name says it)
for name in ["train", "valid", "test"]:
    part = df[df["split"] == name].drop(columns=["split"]).reset_index(drop=True)
    path = PROCESSED_DIR / f"{name}.parquet"
    part.to_parquet(path, index=False)
    print(f"Saved {path.name:<15} {len(part):>9,} loans | default rate {part['default_12m'].mean():.2%} | "
          f"{path.stat().st_size / 1e6:,.0f} MB")

# 4. Record everything later notebooks need
config["splits"] = {
    "train": {"from": "2013-01", "to": TRAIN_END.strftime("%Y-%m")},
    "valid": {"from": "2016-01", "to": VALID_END.strftime("%Y-%m")},
    "test":  {"from": "2017-01", "to": config["target"]["eligible_issue_to"]},
}
config["features"] = {
    "all": FEATURES,
    "numeric": NUMERIC,
    "categorical": CATEGORICAL,
    "structural_missing": STRUCTURAL_MISSING,     # missing means "never happened"
    "excluded_groups": [g for g in COLUMN_GROUPS if g not in ("application", "bureau_core")],
}
with open(INTERIM_DIR / "audit_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("\nConfig updated with splits and feature lists")

Loans with a gap of 5: 232 | their fico_range_low values: [np.float32(845.0)]
-> all in the 845-850 top band: fico_range_high adds nothing, dropped

Features: 62  (59 numeric, 3 categorical)
Categorical: ['home_ownership', 'verification_status', 'purpose']
Saved train.parquet     791,027 loans | default rate 3.11% | 52 MB
Saved valid.parquet     425,618 loans | default rate 4.01% | 29 MB
Saved test.parquet      492,687 loans | default rate 3.74% | 34 MB

Config updated with splits and feature lists


## Findings

1. **Only information available at the moment of application is used.** Of the 151 columns, 39 are only known after the loan starts, 8 are Lending Club's own verdict and pricing, and the rest are admin, free text, location or joint-application fields. That leaves 77 candidates.
2. **Location is excluded on purpose**, so the model can't use where someone lives as a stand-in for race or ethnicity.
3. **The 14 extended credit bureau fields are dropped.** They only exist from late 2015, so in the training years they're almost entirely empty, and keeping them would let the model learn the year instead of the borrower.
4. **Five "months since..." columns are mostly empty because the event never happened**, which was confirmed against columns that count the same events. They're kept for now.
5. **Having a past late payment or bankruptcy barely changes the default rate among approved borrowers.** Lending Club's screening most likely already accounted for it. This is selection bias: the data only contains people who were approved.
6. **The data is clean:** one impossible value, no unseen categories, and credit history lengths that respect Lending Club's 3-year minimum.
7. **62 features across three time-based splits:** train 2013 to 2015 (791,027 loans, 3.11%), validation 2016 (425,618 loans, 4.01%) and test January 2017 to March 2018 (492,687 loans, 3.74%).

**Next:** Notebook 04 explores the 62 features against the 12-month default target, using the training set only.